# BSM L08 — AI-assisted mobile application security assessment

Pracuj na projekcie `student/apps/lesson_h_ai_security` w Android Studio.

W tym notebooku:
- H01 używa [mobsfscan](https://github.com/MobSF/mobsfscan)
- H02 używa [MobSF](https://github.com/MobSF/Mobile-Security-Framework-MobSF)
- H03 używa [SecureBERT2.0-code-vuln-detection](https://huggingface.co/cisco-ai/SecureBERT2.0-code-vuln-detection)
- H04 uruchamia w Pythonie triage między `mobsfscan` i SecureBERT2
- H05 uruchamia w Pythonie patch verifier na `SecurePatchTarget_v1` i `SecurePatchTarget_v2`

Każde zadanie ma dropdown z odpowiedzią i komórkę wysyłki.


In [ ]:
# @title Dane studenta {"run":"auto","vertical-output":true,"display-mode":"form"}
import requests

Student_ID = "" #@param {type:"string"}
Mail = "" #@param {type:"string"}
Grupa = "" #@param {type:"string"}
Link_do_projektu_Kotlin = "" #@param {type:"string"}

BASE_URL = "https://www.duszekjk.com/bsk/"

def wyslij_odpowiedz(task_id, final_answer):
    final_answer = str(final_answer)
    final_answer_size = len(final_answer)
    final_answer_send = final_answer[:600] + "\n" + str(final_answer_size) + " znaków"
    data = {
        "student_id": Student_ID,
        "student_mail": Mail,
        "task": task_id,
        "grupa": Grupa,
        "answer": final_answer_send,
        "share_link": Link_do_projektu_Kotlin,
    }
    url = BASE_URL + "api/submit_answer/"
    r = requests.post(url, json=data, timeout=20)
    print(r.status_code)
    try:
        j = r.json()
        if "points" in j:
            print("Punkty:", j["points"])
        else:
            print(j)
    except Exception:
        print(r.text)
# Komórka pomocnicza: formatowanie i wysyłanie odpowiedzi
answers = {}

def short_text(text, limit=48):
    text = str(text).strip().replace("\n", " ")
    return text[:limit]

def prepare_answer(*parts, limit=220):
    final_answer = "|".join(str(p) for p in parts)
    return final_answer[:limit]

def zapisz_i_wyslij(task_id, final_answer):
    final_answer = str(final_answer)
    answers[task_id] = final_answer
    print(f"Zapisano answers[{task_id}] ({len(final_answer)} znaków)")
    print(final_answer)
    # Wysyłka do backendu zadania
    wyslij_odpowiedz(task_id, final_answer)

def dropdown_choice_to_answer(choice, mapping):
    choice = str(choice).strip()
    if not choice:
        return ""
    letter = choice[:1].upper()
    return mapping.get(letter, "")


# H01 — `mobsfscan` source-code scan

## Teoria
`mobsfscan` analizuje kod źródłowy w sposób deterministyczny: szuka znanych, niebezpiecznych wzorców i dopasowuje je do reguł bezpieczeństwa. W tym labie używasz go do znalezienia hardcoded secreta, plaintext storage, nadmiernego logowania w plikach Kotlin.

Cechy tego typu narzędzia:
- daje powtarzalny wynik dla tego samego kodu,
- jest dobre do wykrywania prostych, znanych antywzorów,
- nie rozumie intencji autora aplikacji,
- nie zastępuje ręcznej weryfikacji, jeśli finding jest niejednoznaczny.

## Co sprawdzasz
Masz przejść katalog `student/apps/lesson_h_ai_security/InsecureNotes/app/src/main/java/com/example/secretlab/insecure/` narzędziem `mobsfscan` i znaleźć przygotowaną podatność w kodzie źródłowym. To zadanie uczy, jak odczytać finding ze skanera i przypisać go do jednego kanonicznego kodu.

## Narzędzie
Jeśli nie masz narzędzia, zainstaluj je w terminalu projektu poleceniem `python3 -m pip install mobsfscan`. Dokumentacja i źródła są na stronie [MobSF/mobsfscan](https://github.com/MobSF/mobsfscan). Potem uruchom `mobsfscan` z katalogu `student/apps/lesson_h_ai_security/InsecureNotes`.

## Krok po kroku
1. Otwórz `student/apps/lesson_h_ai_security/InsecureNotes` w Android Studio.
1. Otwórz terminal w katalogu projektu.
1. Uruchom `mobsfscan app/src/main/java/com/example/secretlab/insecure/`.
1. Odczytaj finding dla `SecretStore.kt`, `LoggingHelper.kt` i `CryptoHelper.kt`.
1. Jeśli masz kilka wyników, wybierz ten z najwyższym znaczeniem bezpieczeństwa: sekret na sztywno, plaintext storage, ekspozycja danych w logach.
1. Jeśli raport pokazuje kilka problemów, wybierz ten, który dotyczy sekretnych stałych w kodzie i ich zapisu w zwykłym storage.
1. Wybierz w dropdownie literę, która najlepiej opisuje wykryty problem.

## Weryfikacja
W formularzu wybierz jedną literę odpowiadającą findingowi. Notebook sam zamieni wybór na kanoniczny kod.


In [ ]:
# @title H01 — Formularz odpowiedzi {"run":"auto","vertical-output":true,"single-column":true,"display-mode":"form"}
choice_h01 = ""  #@param ["", "A - hardcoded secret literal in SecretStore.kt", "B - plaintext token in SharedPreferences", "C - sensitive logging in LoggingHelper.kt", "D - weak crypto in CryptoHelper.kt", "E - exported component in manifest"]
final_answer = prepare_answer(dropdown_choice_to_answer(choice_h01, {"A": "HARDCODED_SECRET", "B": "PLAINTEXT_SHARED_PREFERENCES", "C": "SENSITIVE_LOGGING", "D": "WEAK_CRYPTO", "E": "EXPORTED_COMPONENT"}))
print(final_answer)
zapisz_i_wyslij("H01", final_answer)


# H02 — APK analysis in MobSF

## Teoria
MobSF czyta już zbudowany APK. W tym zadaniu nie patrzysz na kod źródłowy, tylko na to, co znalazło się w paczce: manifest, flagi aplikacji, komponenty eksportowane, konfigurację backupu i cleartext traffic. Raport MobSF jest zbudowany z sekcji, które pomagają szybko wyłapać słabą konfigurację bezpieczeństwa.

Cechy MobSF w tym labie:
- analizuje artefakt po buildzie, a nie pliki robocze w projekcie,
- pokazuje wynik manifestu i flag bezpieczeństwa,
- jest użyteczny do szybkiej oceny konfiguracji APK,
- nie zastępuje porównania z wersją źródłową, jeśli raport jest niepełny.

## Co sprawdzasz
Masz przeanalizować APK `FakeBankLite` w MobSF i wskazać problem widoczny w raporcie manifestu. W tym zadaniu chodzi konkretnie o ustawienie `android:usesCleartextTraffic`.

## Narzędzie
Użyj [MobSF](https://github.com/MobSF/Mobile-Security-Framework-MobSF). To webowa aplikacja z dashboardem, w którym klikniesz `Upload & Analyze`. Jeśli nie widzisz panelu uploadu, to nie jest właściwa instancja narzędzia.

## Krok po kroku
1. Otwórz `student/apps/lesson_h_ai_security/FakeBankLite` w Android Studio.
1. W terminalu w katalogu projektu uruchom `./gradlew assembleDebug`.
1. Otwórz instancję MobSF w przeglądarce i kliknij `Upload & Analyze`.
1. Wgraj plik `app/build/outputs/apk/debug/app-debug.apk`.
1. Poczekaj, aż raport się wygeneruje, i otwórz `Manifest Analysis`.
1. Odszukaj linię `android:usesCleartextTraffic="true"`.
1. Wybierz odpowiedź odnoszącą się do cleartext traffic, nie do backupu ani debugowania.

## Weryfikacja
W formularzu wybierz jedną literę odpowiadającą dokładnie temu ustawieniu manifestu. Notebook sam zamieni wybór na kanoniczny kod.


In [ ]:
# @title H02 — Formularz odpowiedzi {"run":"auto","vertical-output":true,"single-column":true,"display-mode":"form"}
choice_h02 = ""  #@param ["", "A - `android:usesCleartextTraffic=\"true\"` in manifest", "B - `android:allowBackup=\"true\"` in manifest", "C - `android:debuggable=\"true\"` in manifest", "D - exported launcher activity", "E - network security config present"]
final_answer = prepare_answer(dropdown_choice_to_answer(choice_h02, {"A": "CLEARTEXT_TRAFFIC", "B": "BACKUP_ENABLED", "C": "DEBUGGABLE_TRUE", "D": "EXPORTED_COMPONENT", "E": "NO_ISSUE"}))
print(final_answer)
zapisz_i_wyslij("H02", final_answer)


# H03 — `SecureBERT2` security classification

## Teoria
SecureBERT2 bierze krótki fragment kodu, zamienia go na tokeny i zwraca label klasyfikacyjny. To nie jest regułowy skaner. Model ocenia wzorce w tekście kodu i potrafi zareagować na podobieństwo do znanych podatności, ale jego wynik zależy od dokładnego wejścia, wersji modelu i sposobu przygotowania snippetu.

## Uruchomienie w Colab
W notebooku wklej i uruchom ten blok w Colab.
```python
from transformers import pipeline
classifier = pipeline("text-classification", model="cisco-ai/SecureBERT2.0-code-vuln-detection")
snippet = """class SecretStoreSample(private val context: android.content.Context) {
    private val prefs = context.getSharedPreferences("insecure_notes", android.content.Context.MODE_PRIVATE)
    private val demoApiKey = \"demo-api-key-1A2B3C\"

    fun demoApiKey(): String = demoApiKey

    fun saveSessionToken(token: String) {
        prefs.edit().putString("session_token", token).apply()
    }
}"""
result = classifier(snippet, truncation=True)[0]
print(result)
```

## Krok po kroku
1. Otwórz `student/apps/lesson_h_ai_security/InsecureNotes/app/src/main/java/com/example/secretlab/insecure/SecretStore.kt`.
1. Skopiuj dokładnie ten fragment do zmiennej `snippet`.
1. Uruchom blok w Colab.
1. Odczytaj label modelu i sprawdź, czy model widzi podatność w stałej i zapisie do storage.
1. Wybierz odpowiedź z dropdownu.

## Weryfikacja
W formularzu wybierasz jedną literę odpowiadającą labelowi modelu. Notebook sam zamieni wybór na kanoniczny kod.


In [ ]:
# @title H03 — Formularz odpowiedzi {"run":"auto","vertical-output":true,"single-column":true,"display-mode":"form"}
choice_h03 = ""  #@param ["", "A - SecureBERT2 flags the frozen snippet as vulnerable", "B - SecureBERT2 says the snippet is safe", "C - SecureBERT2 returns an unclear result", "D - input was invalid", "E - tool unavailable"]
final_answer = prepare_answer(dropdown_choice_to_answer(choice_h03, {"A": "MODEL_DETECTED_VULNERABLE", "B": "MODEL_RESULT_SAFE", "C": "MODEL_RESULT_UNCLEAR", "D": "MODEL_INPUT_INVALID", "E": "MODEL_TOOL_UNAVAILABLE"}))
print(final_answer)
zapisz_i_wyslij("H03", final_answer)


# H04 — Heterogeneous multi-agent disagreement triage

## Teoria
Heterogeneous multi-agent disagreement triage to układ, w którym regułowy skaner, model klasyfikacyjny i lokalny verifier analizują ten sam fragment kodu i zwracają własne sygnały. W Colab uruchamiasz oba kroki w Pythonie: najpierw skaner, potem klasyfikator SecureBERT2.

## Uruchomienie w Colab
```python
import subprocess
from transformers import pipeline

snippet = """class SecretStoreSample(private val context: android.content.Context) {
    private val prefs = context.getSharedPreferences("insecure_notes", android.content.Context.MODE_PRIVATE)
    private val demoApiKey = \"demo-api-key-1A2B3C\"

    fun demoApiKey(): String = demoApiKey

    fun saveSessionToken(token: String) {
        prefs.edit().putString("session_token", token).apply()
    }
}"""

scanner = subprocess.run(["mobsfscan", "app/src/main/java/com/example/secretlab/insecure/"], capture_output=True, text=True)
classifier = pipeline("text-classification", model="cisco-ai/SecureBERT2.0-code-vuln-detection")
model_result = classifier(snippet, truncation=True)[0]
print(scanner.stdout)
print(model_result)
```

## Co sprawdzasz
Porównujesz wynik `mobsfscan` z H01 i wynik SecureBERT2 z tego samego snippetu. Jeśli skaner wskazuje hardcoded secret, a model zwraca wynik bez podatności, wybierasz kanoniczny kod.

## Weryfikacja
Notebook zapisuje kanoniczny kod `HETEROGENEOUS_MULTI_AGENT_TRIAGE`.


In [ ]:
# @title H04 — Formularz odpowiedzi {"run":"auto","vertical-output":true,"single-column":true,"display-mode":"form"}
choice_h04 = ""  #@param ["", "A - multi-agent verifier flags scanner/model disagreement", "B - scanner only", "C - model only", "D - manual review only"]
final_answer = prepare_answer(dropdown_choice_to_answer(choice_h04, {"A": "HETEROGENEOUS_MULTI_AGENT_TRIAGE", "B": "SCANNER_ONLY", "C": "MODEL_ONLY", "D": "MANUAL_REVIEW_NEEDED"}))
print(final_answer)
zapisz_i_wyslij("H04", final_answer)


# H05 — Agentic patch verification

## Teoria
Agentic patch verification to uruchomiony w Pythonie verifier, który porównuje manifest przed i po poprawce, wyciąga zmienione linie i sprawdza, czy patch usuwa ryzyko `usesCleartextTraffic`. W Colab uruchamiasz ten sam notebook bez ręcznego klikania w edytor diff.

## Uruchomienie w Colab
```python
from pathlib import Path
from difflib import unified_diff
from transformers import pipeline

before = Path("student/apps/lesson_h_ai_security/SecurePatchTarget_v1/app/src/main/AndroidManifest.xml").read_text()
after = Path("student/apps/lesson_h_ai_security/SecurePatchTarget_v2/app/src/main/AndroidManifest.xml").read_text()
diff = "\n".join(unified_diff(before.splitlines(), after.splitlines(), fromfile="v1", tofile="v2"))
verifier = pipeline("text-classification", model="cisco-ai/SecureBERT2.0-code-vuln-detection")
print(diff)
print(verifier(diff, truncation=True)[0])
```

## Co sprawdzasz
Podajesz diff do verifiera i sprawdzasz, czy zmiana `android:usesCleartextTraffic="true"` na `android:usesCleartextTraffic="false"` usuwa problem.

## Weryfikacja
Notebook zapisuje kanoniczny kod `AGENTIC_PATCH_VERIFICATION`.


In [ ]:
# @title H05 — Formularz odpowiedzi {"run":"auto","vertical-output":true,"single-column":true,"display-mode":"form"}
choice_h05 = ""  #@param ["", "A - patch verifier confirms cleartext is removed", "B - backup disabled", "C - debuggable removed", "D - exported component removed", "E - no visible change"]
final_answer = prepare_answer(dropdown_choice_to_answer(choice_h05, {"A": "AGENTIC_PATCH_VERIFICATION", "B": "BACKUP_DISABLED", "C": "DEBUGGABLE_REMOVED", "D": "EXPORTED_COMPONENT_REMOVED", "E": "NO_FIX_DETECTED"}))
print(final_answer)
zapisz_i_wyslij("H05", final_answer)
